# 📊 GigScore — Exploratory Data Analysis

This notebook explores the datasets used in the GigScore alternative credit scoring system:
1. Home Credit Default Risk (filtered for gig-worker profiles)
2. Give Me Some Credit (thin-file applicants)
3. Synthetic India Gig Worker Data (50,000 records)

**Key Questions:**
- What does the target distribution look like? (class imbalance)
- Which features have the most missing values? Why?
- How do income patterns differ between defaulters and non-defaulters?
- What does the gig worker population look like?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('dark_background')
sns.set_palette('viridis')

import sys
sys.path.insert(0, '..')
from src.data.synthetic_generator import generate_synthetic_gig_data

## 1. Data Loading & Shape Overview

In [ ]:
# Generate synthetic data for exploration
df = generate_synthetic_gig_data(n_samples=50000, seed=42)
print(f"Shape: {df.shape}")
print(f"\nColumn Types:\n{df.dtypes.value_counts()}")
df.head()

## 2. Target Variable Analysis (Imbalance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target distribution
df['target'].value_counts().plot.bar(ax=axes[0], color=['#22c55e', '#ef4444'])
axes[0].set_title('Target Distribution')
axes[0].set_xticklabels(['Repaid (0)', 'Defaulted (1)'], rotation=0)

# Percentage
df['target'].value_counts(normalize=True).plot.pie(
    ax=axes[1], autopct='%1.1f%%', colors=['#22c55e', '#ef4444'],
    labels=['Repaid', 'Defaulted']
)
axes[1].set_title('Default Rate')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()
print(f"Default rate: {df['target'].mean():.2%}")

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100)
missing_df = pd.DataFrame({'count': missing, 'percent': missing_pct})
print("Missing values:")
print(missing_df[missing_df['count'] > 0])
print(f"\nNote: Synthetic data has no missing values by design.")
print(f"Home Credit has ~40% missing in EXT_SOURCE_1, ~20% in OCCUPATION_TYPE.")

## 4. Univariate Analysis — Income Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['monthly_earnings_inr'], bins=50, color='#6366f1', alpha=0.7)
axes[0].set_title('Monthly Earnings Distribution')
axes[0].set_xlabel('Monthly Earnings (₹)')

axes[1].hist(df['platform_rating'], bins=30, color='#8b5cf6', alpha=0.7)
axes[1].set_title('Platform Rating Distribution')

axes[2].hist(df['platform_tenure_months'], bins=40, color='#a78bfa', alpha=0.7)
axes[2].set_title('Platform Tenure (months)')

plt.tight_layout()
plt.show()

## 5. Bivariate Analysis — Features vs Default Rate

In [ ]:
# Key Insight 1: Income vs Default Rate
df['income_bucket'] = pd.qcut(df['monthly_earnings_inr'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
income_default = df.groupby('income_bucket', observed=False)['target'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
income_default.plot.bar(ax=axes[0], color='#ef4444', alpha=0.8)
axes[0].set_title('Default Rate by Income Bucket')
axes[0].set_ylabel('Default Rate')
axes[0].tick_params(axis='x', rotation=45)

# Key Insight 2: Platform Tenure vs Default Rate
df['tenure_bucket'] = pd.cut(df['platform_tenure_months'], bins=[0, 6, 12, 24, 48, 100], 
                              labels=['0-6m', '6-12m', '1-2yr', '2-4yr', '4+yr'])
tenure_default = df.groupby('tenure_bucket', observed=False)['target'].mean()
tenure_default.plot.bar(ax=axes[1], color='#f59e0b', alpha=0.8)
axes[1].set_title('Default Rate by Platform Tenure')
axes[1].set_ylabel('Default Rate')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_with_target = df[numeric_cols].corr()['target'].drop('target').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
corr_with_target.plot.barh(ax=ax, color=['#ef4444' if x > 0 else '#22c55e' for x in corr_with_target])
ax.set_title('Feature Correlation with Default (Target)')
ax.set_xlabel('Pearson Correlation')
ax.axvline(x=0, color='white', linewidth=0.5)
plt.tight_layout()
plt.show()

## 7. Key Insights Summary

1. **Income vs Default**: Lower income gig workers have significantly higher default rates
2. **Platform Tenure**: Workers with 4+ years of tenure have ~50% lower default rates than new workers
3. **Income Volatility**: The strongest predictor — volatile income directly increases default risk
4. **Savings Account**: Having a savings account reduces default probability (financial buffer)
5. **Platform Rating**: Higher-rated workers default less — rating is a proxy for reliability